# EDA

Preguntas sobre el tracking crudo: qué hay en el archivo, la velocidad del balón, cuántos jugadores se ven.

Los fallos de una solución y la comparación con la referencia se miran en `06_revision.ipynb`, que carga el `frame_df.csv` escrito por `05_algoritmo.ipynb`. Este notebook no corre el algoritmo.

`frame_id` es el frame del vídeo y del CSV que se entrega. Coincide con el índice de la fila. `tracking_frame_id` es el id original del archivo de tracking, y vale `frame_id + 1`.


In [1]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
from kloppy import metrica


import games
import time

paths = games.asset_paths(source='tactical')


# Summary of Changes and observations


- Observation: Ball speed is present in atd file, has some entries with values but in frame_df all values are empty. Kloppy metrica doesn't include the field, we created own reader for speed and merged it with frame_df
- Observation: Players have: x, y, d and s. D field is not present in file and all values are empty.
- Observation: Players info we have 22 players on the pitch. But tracker has 46 trackers. As commented in the data description:  "Slots appear, vanish and swap identity as the tracker loses and re-acquires people. Expect around 22 tracked objects during play — the mean over graded frames is 20.5, and it occasionally exceeds 22 when the tracker duplicates someone. You get positions and team membership, nothing more, so any analysis you build has to be collective rather than per-player". Strategies to relign players by positions, check duplicates when we have more than 22 players might be usefull. Also file marks slots:  24 y el 46 as Goalkeeper. Others as unknown. 


# Conclusions


- Ball speed read from the file doesn't seem to apport extra information. Ball speed ms is computed from ball position comparing, prev frame position if exist with current.


## Match Related Info

- https://www.fcbarcelona.es/es/partidos/116220/fc-barcelona-athletic-club-la-liga-2024-2025
- https://www.laliga.com/partido/temporada-2024-2025-laliga-ea-sports-fc-barcelona-athletic-club-2


## Radu review ideas


- Duplicated pos for players are just 2000 frames or less during match, review if this can help us improve
- Ball speed was geenrated it from ball position from on frame compared with previous. So missin balls if we have the speed we can check if its out of the field. just an estimator. Detectar fueras de campo de la pelota.


## Read full dataset evaluation:


In [2]:
t0 = time.perf_counter()
dataset = metrica.load_tracking_epts(
    meta_data=paths["atd_meta"],
    raw_data=paths["atd_raw"],
)
elapsed = time.perf_counter() - t0
print("%d frames in %.1f s" % (len(dataset.records), elapsed))


199319 frames in 26.7 s


In [3]:
t1 = time.perf_counter()
frame_df = dataset.to_df()
elapsed = time.perf_counter() - t1
print("%d frames in %s s" % (frame_df.shape[0], elapsed))


In [4]:
# Kloppy's frame_id starts at 1. The submission CSV and the video start at 0.
# After this cell, frame_id is the deliverable frame and matches the row index.
# tracking_frame_id keeps the original id. It is frame_id + 1.
frame_df["tracking_frame_id"] = frame_df["frame_id"].astype(int)
frame_df["frame_id"] = frame_df["tracking_frame_id"] - 1
frame_df = frame_df.reset_index(drop=True)
frame_df[["frame_id", "tracking_frame_id"]].head(3)


## Ball speed


In [5]:
def load_ball_speed(raw_path, n_frames):
    """Ball speed from the raw ATD, in the unit the XML declares (kph).

    The ball is the third ':' field of each line, and speed is its third comma
    value. Kloppy drops that value, so this reads the text directly. Index i is
    video frame i. NaN where the file has no speed.
    """
    speed = np.full(n_frames, np.nan)
    with open(raw_path) as handle:
        for line in handle:
            frame, _, ball = line.split(":", 2)
            value = ball.rstrip("\n").split(",")[2]
            if value != "NaN":
                speed[int(frame) - 1] = float(value)
    return speed


ball_speed = load_ball_speed(paths["atd_raw"], games.GAMES[games.GAME]["n_video_frames"])
print("numeric speeds: %d of %d" % (np.isfinite(ball_speed).sum(), len(ball_speed)))

# frame_id is the video frame. ball_speed[i] is the same frame.
frame_df["ball_speed"] = ball_speed[frame_df["frame_id"].to_numpy()]
frame_df["ball_speed"].value_counts(dropna=False)


In [6]:
frame_df


In [7]:
round((frame_df[frame_df["ball_x"].isna() == False].shape[0] / frame_df.shape[0]) * 100, 2)


In [8]:
round((frame_df[frame_df["ball_y"].isna() == False].shape[0] / frame_df.shape[0]) * 100, 2)


In [9]:
frame_df[(frame_df["ball_x"].isna() == False) & (frame_df["ball_y"].isna() != False)]


In [10]:
frame_df[(frame_df["ball_x"].isna() != False) & (frame_df["ball_y"].isna() == False)]


In [11]:
frame_df[frame_df["ball_speed"].isna() == False]


In [12]:
round((frame_df[frame_df["ball_speed"].isna() == False].shape[0] / frame_df.shape[0]) * 100, 2)


In [13]:
frame_df["ball_speed"].value_counts(dropna=False)


In [14]:
frame_df[["ball_speed"]].sort_values(by="ball_speed", ascending=False)


In [15]:
frame_df[(frame_df["ball_x"].isna() == False) & 
        (frame_df["ball_y"].isna() == False) & 
        (frame_df["ball_speed"].isna() == False)]


In [16]:
subset = frame_df[
    frame_df["ball_x"].notna()
    & frame_df["ball_y"].notna()
    & frame_df["ball_speed"].notna()
]
plt.figure(figsize=(12, 4))
plt.scatter((subset["frame_id"]) / 25, subset["ball_speed"], s=2)
plt.xlabel("segundos")
plt.ylabel("ball_speed (km/h)")
plt.show()


In [17]:
subset = frame_df[
    frame_df["ball_x"].notna()
    & frame_df["ball_y"].notna()
    & frame_df["ball_speed"].notna()
]
plt.figure(figsize=(12, 4))
plt.scatter((subset["frame_id"]) / (25 * 60), subset["ball_speed"], s=2)
plt.xlabel("minutos")
plt.ylabel("ball_speed (km/h)")
plt.show()


## Compute own ball speed


In [18]:
L, W, fps = 105.3, 68.0, 25


In [19]:
dx = frame_df["ball_x"].diff() * L
dy = frame_df["ball_y"].diff() * W
seen = frame_df["ball_x"].notna() & frame_df["ball_y"].notna()


In [20]:
frame_df["ball_speed_mps"] = np.hypot(dx, dy) * fps
frame_df.loc[~(seen & seen.shift(fill_value=False)), "ball_speed_mps"] = np.nan


In [21]:
def plot_speed(df, title):
    part = df.loc[df["ball_speed_mps"].notna()]
    video_frame = part["frame_id"]
    fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharey=True)
    for ax, x, label in zip(
        axes,
        (video_frame, video_frame / 25, video_frame / (25 * 60)),
        ("frame", "segundos", "minutos"),
    ):
        ax.scatter(x, part["ball_speed_mps"], s=2)
        ax.set_xlabel(label)
        ax.set_ylabel("m/s")
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()


plot_speed(frame_df, "archivo entero")

# Saque inicial de la primera parte (11185) hasta el último frame puntuado (190197).
# El descanso queda en medio y casi no tiene puntos.
video_frame = frame_df["frame_id"]
play = frame_df[video_frame.between(11185, 190197)]
plot_speed(play, "del saque inicial al final del partido")


## Players tracker


In [22]:
frame_df


In [23]:
x_cols = [f"{i}_x" for i in range(1, 47)]
s_cols = [f"{i}_s" for i in range(1, 47)]


In [24]:
frame_df["n_players"] = frame_df[x_cols].notna().sum(axis=1)
frame_df["n_players_speed"] = frame_df[s_cols].notna().sum(axis=1)


In [25]:
# cut just match:
video_frame = frame_df["frame_id"]
play = frame_df[video_frame.between(11185, 190197)]


In [26]:
plt.figure(figsize=(12, 4))
plt.plot(video_frame.loc[play.index] / (25 * 60), play["n_players"], lw=0.4)
plt.axhline(22, color="red", lw=1, label="22 en el campo")
plt.xlabel("minutes")
plt.ylabel("players with pos tracked")
plt.legend()
plt.show()


In [27]:
n = len(play)
print("match frames: %d" % n)
print("fewer than 22: %d (%.1f%%)" % ((play["n_players"] < 22).sum(), 100 * (play["n_players"] < 22).mean()))
print("exactly 22: %d (%.1f%%)" % ((play["n_players"] == 22).sum(), 100 * (play["n_players"] == 22).mean()))
print("more than 22: %d (%.1f%%)" % ((play["n_players"] > 22).sum(), 100 * (play["n_players"] > 22).mean()))
print("22 or more: %d (%.1f%%)" % ((play["n_players"] >= 22).sum(), 100 * (play["n_players"] >= 22).mean()))
print("mean players with a position: %.2f" % play["n_players"].mean())
print("mean players with a speed: %.2f" % play["n_players_speed"].mean())


From players tracked positions at first glance we can understand something happend on minutes 30 aprox, 40, 50, and more pronunciated at 60 minute and 110 aprox.


In [28]:
frame_df


## Dónde seguir

Una regla nueva se escribe en `05_algoritmo.ipynb`. Después de ejecutarla y de pasar `run_evaluation.py`, los frames concretos se miran en `06_revision.ipynb`.
